# Inference speed: HF Transformers vs vLLM

Benchmark single-prompt latency and throughput on the same
Llama-3.2-1B-Instruct checkpoint across:

1. HF Transformers (eager generation)
2. vLLM with `gpu_memory_utilization=0.3`
3. vLLM with `gpu_memory_utilization=0.7`

Decoding is stochastic (`do_sample=True` / `temperature > 0`) on
both backends, with matching `temperature` and `top_p`. Each timed
iteration uses a fresh seed `base_seed + i` — outputs differ from
run to run but are reproducible across re-executions of the cell.
Token counts vary per iteration since the sampled completions hit
EOS at different points; throughput (tokens / second) is the
robust metric to compare.

Each backend includes one untimed warmup pass to exclude cudagraph
capture / JIT compilation from the latency.

Note: `gpu_memory_utilization` sets the total GPU memory budget
for both model weights and the KV cache. On a single prompt the
two vLLM settings should land within noise of each other; the
difference shows up under concurrent batching (more sequences live
in cache at once).

## Setup

In [ ]:
import gc
import logging
import os
import sys
import time
import warnings

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
warnings.filterwarnings("ignore")

sys.path.append("..")
from unittests.notebook_utils import gpu_mem_used_gb, measure_inference

In [2]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

dataset_dir = base_dir + "/prm800k/math_splits"

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
llm_dir = base_dir + "Llama3.2-1B-Instruct"
# llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [3]:
# Benchmark prompt + decoding config
prompt = (
    r'If $f(x) = \frac{3x-2}{x-2}$, what is the value of '
    r'$f(-2) + f(-1) + f(0)$? Express your answer as a common fraction.'
)
max_new_tokens = 1024
num_runs = 10

# Stochastic decoding — same params on both backends for a fair comparison
temperature = 0.8
top_p = 0.95
base_seed = 123    # iteration i uses seed = base_seed + i

## HF Transformers (baseline)

In [4]:
tokenizer = AutoTokenizer.from_pretrained(llm_dir)
model_hf = AutoModelForCausalLM.from_pretrained(
    llm_dir,
    dtype="float16",
    device_map="cuda:0",
)
model_hf.eval()

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')
print(model_hf.dtype)

#--- GPU memory used: 2.70 GB
torch.float16


In [5]:
latency_hf, throughput_hf, avg_tokens_hf, text_hf = measure_inference(
    "hf", model_hf, tokenizer, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"HF Transformers   - latency: {latency_hf:.4f}s, "
    f"throughput: {throughput_hf:.2f} tok/s, "
    f"avg tokens: {avg_tokens_hf:.1f}"
)

HF Transformers   - latency: 8.5163s, throughput: 58.98 tok/s, avg tokens: 502.3


In [6]:
# Free HF before loading vLLM so they don't fight over the GPU
del model_hf
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

#--- GPU memory used: 2.68 GB


## vLLM with `gpu_memory_utilization=0.3`

Small total memory budget — leaves limited headroom for the KV
cache after model weights are loaded.

In [7]:
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.3,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

INFO 06-12 13:02:16 [utils.py:233] non-default args: {'dtype': 'float16', 'seed': 123, 'max_model_len': 5000, 'gpu_memory_utilization': 0.3, 'disable_log_stats': True, 'model': '/groups/chichengz/tnn/datasets/Llama3.2-1B-Instruct'}
INFO 06-12 13:02:16 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 06-12 13:02:16 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 06-12 13:02:16 [model.py:1582] Using max model len 5000
INFO 06-12 13:02:16 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-12 13:02:16 [vllm.py:775] Asynchronous scheduling is enabled.
WARNING 06-12 13:02:19 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.34s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.34s/it]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 47.97it/s]
Capturing CUD

INFO 06-12 13:02:44 [llm.py:391] Supported tasks: ['generate']
#--- GPU memory used: 12.83 GB


In [8]:
latency_v03, throughput_v03, avg_tokens_v03, text_v03 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.3  - latency: {latency_v03:.4f}s, "
    f"throughput: {throughput_v03:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v03:.1f}"
)

vLLM gpu_mem=0.3  - latency: 1.9383s, throughput: 253.10 tok/s, avg tokens: 490.6


In [9]:
# Free the first vLLM engine before reloading at a higher pool size
del llm_vllm
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

[rank0]:[W612 13:03:05.985481210 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


#--- GPU memory used: 2.68 GB


## vLLM with `gpu_memory_utilization=0.7`

Large total memory budget — more GPU memory reserved for the KV
cache. For a single prompt this should match the 0.3 setting
within noise; the difference shows up under concurrent batching
(more sequences live in cache at once).

In [10]:
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.7,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

INFO 06-12 13:03:06 [utils.py:233] non-default args: {'dtype': 'float16', 'seed': 123, 'max_model_len': 5000, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': '/groups/chichengz/tnn/datasets/Llama3.2-1B-Instruct'}
INFO 06-12 13:03:06 [model.py:533] Resolved architecture: LlamaForCausalLM
WARNING 06-12 13:03:06 [model.py:1920] Casting torch.bfloat16 to torch.float16.
INFO 06-12 13:03:06 [model.py:1582] Using max model len 5000
INFO 06-12 13:03:06 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=8192.


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.37s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.37s/it]

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 47.92it/s]
Capturing CUD

INFO 06-12 13:03:27 [llm.py:391] Supported tasks: ['generate']
#--- GPU memory used: 25.52 GB


In [11]:
latency_v07, throughput_v07, avg_tokens_v07, text_v07 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.7  - latency: {latency_v07:.4f}s, "
    f"throughput: {throughput_v07:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v07:.1f}"
)

vLLM gpu_mem=0.7  - latency: 1.9387s, throughput: 253.05 tok/s, avg tokens: 490.6


## Summary

In [12]:
header = f"{'Backend':<22}{'Latency (s)':>14}{'Tok/s':>12}{'Avg tok':>12}"
print(header)
print('-' * len(header))
rows = [
    ('HF Transformers',  latency_hf,  throughput_hf,  avg_tokens_hf),
    ('vLLM gpu_mem=0.3', latency_v03, throughput_v03, avg_tokens_v03),
    ('vLLM gpu_mem=0.7', latency_v07, throughput_v07, avg_tokens_v07),
]
for name, lat, tput, ntok in rows:
    print(f"{name:<22}{lat:>14.4f}{tput:>12.2f}{ntok:>12.1f}")

Backend                  Latency (s)       Tok/s     Avg tok
------------------------------------------------------------
HF Transformers               8.5163       58.98       502.3
vLLM gpu_mem=0.3              1.9383      253.10       490.6
vLLM gpu_mem=0.7              1.9387      253.05       490.6
